In [1]:
import datetime
import pandas as pd
import requests
import os
import sys
import json
from threading import Thread
# # uppend upper dir
# upper_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
# with open(upper_dir + "/trade_core_config.json") as f:
#     config = json.load(f)

with open("/home/trade_core/trade_core_config.json") as f:
    config = json.load(f)

node = config['node']
prod = config['node_settings'][node]['prod']

class AcwApi:
    def __init__(self, prod=prod):
        if prod is False:
            self.verify = False
            self.url = config['acw_setting']['dev_url']
        else:
            self.verify = True
            self.url = config['acw_setting']['prod_url']
        self.message_url = "messagecore/messages/"
        self.node_url = "tradecore/nodes/"
        self.referral_commission_url = "referral/referral-commission/"
        self.deposit_balance = "users/deposit-balance/"
        self.deposit_history = "users/deposit-history/"

    def get_message(self, id=None, type=None):
        url = self.url + self.message_url
        if type is not None:
                params = {
                    "type": type
                }
        else:
            params = None
        if id is not None:
            fetch_list = False
            response = requests.get(url + str(id) + "/", verify=self.verify, params=params)
        else:
            fetch_list = True
            response = requests.get(url, verify=self.verify, params=params)
        if response.status_code == 200:
            if fetch_list:
                return pd.DataFrame(response.json()["results"])
            else:
                return pd.DataFrame([response.json()])
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def create_message(self, telegram_chat_id, title, origin, type, content=None, remark=None, code=None, sent=False, send_times=1, send_term=1):
        url = self.url + self.message_url
        new_message_dict = {
            "datetime": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S"),
            "telegram_chat_id": telegram_chat_id,
            "title": title,
            "origin": origin,
            "type": type,
            "content": content,
            "remark": remark,
            "code": code,
            "sent": sent,
            "send_times": send_times,
            "send_term": send_term
        }
        response = requests.post(url, json=new_message_dict, verify=self.verify)
        if response.status_code == 201:
            return response.json()
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def update_read_message(self, id):
        url = self.url + self.message_url
        response = requests.patch(url + str(id) + "/", json={"read": True}, verify=self.verify)
        if response.status_code == 200:
            return response.json()
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def create_message_thread(self, telegram_chat_id, title, origin, type, content=None, remark=None, code=None, sent=False, send_times=1, send_term=1):
        t = Thread(target=self.create_message, args=(telegram_chat_id, title, origin, type, content, remark, code, sent, send_times, send_term), daemon=True)
        t.start()

    def delete_message(self, id):
        url = self.url + self.message_url
        response = requests.delete(url + str(id) + "/", verify=self.verify)
        if response.status_code == 204:
            return True
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def get_node(self, id=None):
        url = self.url + self.node_url
        if id is not None:
            fetch_list = False
            response = requests.get(url + str(id) + "/", verify=self.verify)
        else:
            fetch_list = True
            response = requests.get(url, verify=self.verify)
        if response.status_code == 200:
            if fetch_list:
                return pd.DataFrame(response.json()["results"])
            else:
                return pd.DataFrame([response.json()])
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def get_referral_commission(self, user, trade_uuid, initial_profit, target_market_code, origin_market_code, apply_to_deposit=False):
        url = self.url + self.referral_commission_url
        response = requests.get(url, params={"user": user, "trade_uuid": trade_uuid, "initial_profit": initial_profit, "target_market_code": target_market_code, "origin_market_code": origin_market_code}, verify=self.verify)
        if response.status_code == 200:
            return response.json()
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def get_deposit_history(self, user):
        url = self.url + self.deposit_history
        params = {
            "user": user
        }
        response = requests.get(url, params=params, verify=self.verify)
        if response.status_code == 200:
            return pd.DataFrame(response.json()["results"])
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def create_deposit_history(self, user, balance, change, txid, type, pending, registered_datetime=datetime.datetime.utcnow()):
        url = self.url + self.deposit_history
        new_deposit_history_dict = {
            "user": user,
            "balance": balance,
            "change": change,
            "txid": txid,
            "type": type,
            "pending": pending,
            "registered_datetime": registered_datetime.strftime("%Y-%m-%dT%H:%M:%S")
        }
        response = requests.post(url, json=new_deposit_history_dict, verify=self.verify)
        if response.status_code == 201:
            return response.json()
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)
        
    def get_deposit_balance(self, user=None):
        url = self.url + self.deposit_balance
        params = {
            "user": user
        }
        response = requests.get(url, params=params, verify=self.verify)
        if response.status_code == 200:
            return pd.DataFrame(response.json()["results"])
        else:
            raise Exception("Error: " + str(response.status_code) + "\n" + response.text)

In [2]:
import time

In [2]:
acw_api = AcwApi()

In [3]:
df = acw_api.get_deposit_balance()
negative_df = df[df['balance'].astype('float')<0]
negative_df['user'].tolist()

/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-test.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[]

In [13]:
acw_api.update_read_message(59704)

/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-test.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'id': 59704,
 'datetime': '2024-05-10T12:42:06+0000',
 'telegram_bot_username': 'arbicrypto_dev_bot',
 'telegram_chat_id': 1390695186,
 'title': '업비트 SHORT 성공',
 'content': '거래ID: 1의 업비트 KRW-XRP SHORT거래가 정상적으로 진행되었습니다.',
 'remarks': None,
 'origin': 'TEST',
 'type': 'info',
 'code': None,
 'sent': True,
 'send_times': 1,
 'send_term': 1,
 'read': False}

In [14]:
acw_api.get_message(type=None)

/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-test.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,id,datetime,telegram_bot_username,telegram_chat_id,title,content,remarks,origin,type,code,sent,send_times,send_term,read
0,59704,2024-05-10T12:42:06+0000,arbicrypto_dev_bot,1390695186,업비트 SHORT 성공,거래ID: 1의 업비트 KRW-XRP SHORT거래가 정상적으로 진행되었습니다.,None,TEST,info,None,True,1,1,False
1,59703,2024-05-10T12:42:05+0000,arbicrypto_dev_bot,1390695186,청산 알람,청산 알람! 바이낸스 XRPUSDT 0.001개가 강제청산되었습니다.,None,TEST,warning,None,True,5,5,True
2,59702,2024-05-10T12:42:05+0000,arbicrypto_dev_bot,1390695186,청산 자동정리,"마진콜 모니터링 설정에 따라, 자동거래에 진입되어 있는 UPBIT의 포지션을 자동정...",None,TEST,info,None,True,1,1,True
3,59701,2024-05-10T12:37:51+0000,arbicrypto_dev_bot,1390695186,진입거래 성공,진입거래 성공\n거래ID: 1의 거래가 정상적으로 진행되었습니다.\n진입프리미엄: ...,None,trade_core_dev,info,None,True,1,1,True
4,59699,2024-05-10T12:37:50+0000,arbicrypto_dev_bot,1390695186,바이낸스 SHORT 성공,거래ID: 1의 바이낸스 XRPUSDT SHORT거래가 정상적으로 진행되었습니다.,None,trade_core_dev,info,None,True,1,1,True
5,59700,2024-05-10T12:37:50+0000,arbicrypto_dev_bot,1390695186,거래ID: 1(XRP/KRW) 프리미엄 하향돌파,거래ID: 1(XRP/KRW) 프리미엄 하향돌파\nUPBIT_SPOT/KRW:BIN...,None,trade_core_dev,info,None,True,1,1,True
6,59698,2024-05-10T12:37:50+0000,arbicrypto_dev_bot,1390695186,업비트 LONG 성공,거래ID: 1의 업비트 KRW-XRP LONG거래가 정상적으로 진행되었습니다.,None,trade_core_dev,info,None,True,1,1,True
7,59695,2024-05-10T12:31:40+0000,arbicrypto_dev_bot,1390695186,업비트 SHORT 성공,거래ID: 1의 업비트 KRW-XRP SHORT거래가 정상적으로 진행되었습니다.,None,TEST,info,None,True,1,1,True
8,59693,2024-05-10T12:31:39+0000,arbicrypto_dev_bot,1390695186,청산 알람,청산 알람! 바이낸스 XRPUSDT 0.001개가 강제청산되었습니다.,None,TEST,warning,None,True,5,5,True
9,59694,2024-05-10T12:31:39+0000,arbicrypto_dev_bot,1390695186,청산 자동정리,"마진콜 모니터링 설정에 따라, 자동거래에 진입되어 있는 BINANCE와 UPBIT의...",None,TEST,info,None,True,1,1,True


In [4]:
user = "ab2a37d8-65cf-4d23-8168-19e2e26e61d4"
trade_uuid = "700469e0-3bea-426e-ae3b-318f1a8c1de7"
initial_profit = 1000
target_market_code = "UPBIT_SPOT/KRW"
origin_market_code = "BINANCE_USD_M/USDT"

referral_commission_df = acw_api.get_referral_commission(user, trade_uuid, initial_profit, target_market_code, origin_market_code)
referral_commission_df

/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-test.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'trade_uuid': '700469e0-3bea-426e-ae3b-318f1a8c1de7',
 'changes': [{'user': 'ab2a37d8-65cf-4d23-8168-19e2e26e61d4',
   'change': 500.0,
   'type': 'NET_PROFIT_FROM_TRADE'},
  {'user': 'ab2a37d8-65cf-4d23-8168-19e2e26e61d4',
   'change': -500.0,
   'type': 'FEE'},
  {'user': '7a867503-3ff1-4947-b190-d614690bc7b4',
   'change': 45.0,
   'type': 'COMMISSION',
   'commission_from': 'ab2a37d8-65cf-4d23-8168-19e2e26e61d4'},
  {'user': 'b0275c8b-a8b3-47d6-9bc9-fe06363fe232',
   'change': 5.0,
   'type': 'COMMISSION',
   'commission_from': '7a867503-3ff1-4947-b190-d614690bc7b4'}]}

In [7]:
acw_api.get_deposit_balance()

/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-test.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,id,user,balance,last_update
0,1,ab2a37d8-65cf-4d23-8168-19e2e26e61d4,10000.00,2024-04-19T05:35:13+0000


In [37]:
acw_api.get_deposit_history("e16b3bdc-301b-4512-be3b-f0e60cef7004")

/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-test.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


https://acw-test.orbitholdings.org/api/users/deposit-history/?user=e16b3bdc-301b-4512-be3b-f0e60cef7004


""


In [5]:
# node = 'trade_core_dev'
# all_node_df = acw_api.get_node()
# node_df = all_node_df[all_node_df['name']==node]
# if len(node_df) != 1:
#     raise Exception(f"Node not found or not unique, len(node_df)={len(node_df)}")
# enabled_market_code_services = node_df['market_code_services'].values[0]

/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-test.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [14]:
# start = time.time()
# acw_api.create_message(telegram_chat_id=1390695186, title="test", origin="test", type="info", content="test", remark="test", code=None, sent=False, send_times=1, send_term=1)
# print(time.time()-start)

0.10544180870056152


/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-test.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [12]:
response.json()

{'results': [{'uuid': '342b6270-cfd9-4676-90e4-125f67e4c03c',
   'email': 'superuser@halo-soft.net',
   'username': 'superuser',
   'first_name': 'Super',
   'last_name': 'User',
   'role': 'internal',
   'is_active': True,
   'date_joined': '2023-10-04T04:18:11+0000',
   'profile': {'referral': None,
    'level': 1,
    'points': 0,
    'picture': None,
    'user': '342b6270-cfd9-4676-90e4-125f67e4c03c'},
   'favorite_assets': [],
   'socialapps': [],
   'telegram_chat_id': None,
   'trade_config_allocations': []},
  {'uuid': '2c280bd2-6289-4e24-b1ee-5b7c5786265e',
   'email': 'ceo@halo-soft.net',
   'username': '헤일로테스트',
   'first_name': 'Chang-un',
   'last_name': 'Park',
   'role': 'user',
   'is_active': True,
   'date_joined': '2023-10-20T02:23:59+0000',
   'profile': {'referral': None,
    'level': 1,
    'points': 0,
    'picture': 'https://lh3.googleusercontent.com/a/ACg8ocIerr63_JO6LcR_Y4tWhz6pDV69zj8hMuvcc2eM_4fQ=s96-c',
    'user': '2c280bd2-6289-4e24-b1ee-5b7c5786265e'},
 

In [17]:
acw_api.get_node()

/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-dev.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,id,name,url,description,max_user_count,market_code_services
0,1,trade_core_dev,http://221.148.128.205:20081,changun's test tradecore server,300,"[UPBIT_SPOT/KRW:BINANCE_USD_M/USDT, BITHUMB_SP..."


In [34]:
node = 'trade_core_dev'

all_node_df = acw_api.get_node()
node_df = all_node_df[all_node_df['name']==node]
if len(node_df) != 1:
    raise Exception(f"Node not found or not unique, len(node_df)={len(node_df)}")
node_df['market_code_services'].values[0]



/usr/local/lib/python3.9/site-packages/urllib3/connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'acw-dev.orbitholdings.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


['UPBIT_SPOT/KRW:BINANCE_USD_M/USDT', 'BITHUMB_SPOT/KRW:BINANCE_SPOT/USDT']